# LoRA (Low-Rank Adaptation) with blox

This notebook demonstrates how to implement LoRA fine-tuning with blox.

LoRA is a parameter-efficient fine-tuning technique that freezes the pretrained model weights and injects trainable low-rank decomposition matrices. Instead of fine-tuning `W`, we compute `W + A @ B` where `A` and `B` are small matrices.

Key benefits:
- **Memory efficient**: Only LoRA parameters need gradients
- **Modular**: Can add/remove adapters without changing base model
- **Stackable**: Multiple adapters can be combined

Reference: [LoRA: Low-Rank Adaptation of Large Language Models](https://arxiv.org/abs/2106.09685)

In [1]:
import sys

sys.path.insert(0, '../src')

import blox as bx
import jax
import jax.numpy as jnp

## 1. Define an MLP Module

First, let's create a reusable MLP module that we'll later adapt with LoRA.

In [2]:
class MLP(bx.Module):
  """Multi-layer perceptron with configurable layers and activation."""

  def __init__(
      self,
      graph: bx.Graph,
      output_sizes: list[int],
      rng: bx.Rng,
      activation=jax.nn.relu,
  ):
    super().__init__(graph)
    self.activation = activation
    self.layers = []
    for i, size in enumerate(output_sizes):
      name = f'hidden{i}' if i < len(output_sizes) - 1 else 'output'
      self.layers.append(bx.Linear(graph.child(name), size, rng=rng))

  def __call__(self, params, x):
    for i, layer in enumerate(self.layers):
      x, params = layer(params, x)
      # Apply activation to all but the last layer.
      if i < len(self.layers) - 1:
        x = self.activation(x)
    return x, params


print('MLP module defined!')

MLP module defined!


In [3]:
# Create the model.
graph = bx.Graph('net')
rng = bx.Rng(graph.child('rng'))

model = MLP(
    graph.child('mlp'),
    output_sizes=[64, 32, 10],
    rng=rng,
    activation=jax.nn.relu,
)

# Initialize the model.
x = jnp.ones((4, 16))
params = rng.seed(bx.Params(), seed=42)
_, params = model(params, x)
params = params.locked()

print('Model initialized!')
print(f'Total parameters: {len(params)}')

Model initialized!
Total parameters: 8


## 2. Explore the Graph

blox provides `Graph.walk()` to iterate over all modules in the graph. This is useful for finding which layers to apply LoRA to.

In [ ]:
# Walk the graph to see all modules.
print('Modules in graph:')
for path, module in graph.walk():
  print(f'  {path}: {module!r}')

Modules in graph:
  ('net', 'rng'): Rng(auto_fold_in_axes=True)
  ('net', 'mlp'): MLP(output_sizes=[64, 32, 10], rng=Rng(auto_fold_in_axes=True), activation=<jax._src.custom_derivatives.custom_jvp object at 0x7aade2b336b0>)
  ('net', 'mlp', 'hidden0'): Linear(output_size=64, rng=Rng(auto_fold_in_axes=True), use_bias=True, kernel_init=<function variance_scaling.<locals>.init at 0x7aade29d76a0>, bias_init=<function zeros at 0x7aadf813ade0>, kernel_metadata=None, bias_metadata=None)
  ('net', 'mlp', 'hidden1'): Linear(output_size=32, rng=Rng(auto_fold_in_axes=True), use_bias=True, kernel_init=<function variance_scaling.<locals>.init at 0x7aade29d76a0>, bias_init=<function zeros at 0x7aadf813ade0>, kernel_metadata=None, bias_metadata=None)
  ('net', 'mlp', 'output'): Linear(output_size=10, rng=Rng(auto_fold_in_axes=True), use_bias=True, kernel_init=<function variance_scaling.<locals>.init at 0x7aade29d76a0>, bias_init=<function zeros at 0x7aadf813ade0>, kernel_metadata=None, bias_metadata=

## 3. The LoRA Pattern

LoRA replaces `y = x @ W` with `y = x @ W + x @ A @ B` where:
- `W` is frozen (original weights)
- `A` has shape `(in_features, rank)` - initialized with Gaussian
- `B` has shape `(rank, out_features)` - initialized to zeros

**Initialization**: Following the [original paper](https://arxiv.org/abs/2106.09685), A is initialized with a Gaussian distribution and B is initialized to zeros. This ensures `A @ B = 0` at the start, so the model begins with its original pretrained behavior.

**Scaling**: The output is scaled by `alpha / rank` to stabilize training across different rank values.

In [5]:
def apply_lora(layer: bx.Linear, rank: int = 4, alpha: float | None = None):
  """Applies LoRA to a Linear layer by monkey-patching get_param.

  Args:
    layer: The Linear layer to adapt.
    rank: The rank of the low-rank matrices.
    alpha: Scaling factor. Defaults to 2 * rank (common heuristic).
  """
  if alpha is None:
    alpha = 2 * rank

  # Save the original get_param.
  original_get_param = layer.get_param

  def lora_get_param(params, name, shape=None, init=None, rng=None, **kwargs):
    # Get the original parameter.
    value, params = original_get_param(
        params, name, shape, init, rng=rng, **kwargs
    )

    # Only apply LoRA to the kernel.
    if name != 'kernel':
      return value, params

    in_features, out_features = value.shape

    # LoRA initialization (from the original paper):
    # - A: Gaussian with std = 1/sqrt(rank) for stable gradients.
    # - B: Zeros, so LoRA starts as identity (A @ 0 = 0).
    lora_a, params = original_get_param(
        params,
        'lora_a',
        (in_features, rank),
        jax.nn.initializers.normal(stddev=1.0 / jnp.sqrt(rank)),
        rng=rng,
    )
    lora_b, params = original_get_param(
        params,
        'lora_b',
        (rank, out_features),
        jax.nn.initializers.zeros,
        rng=rng,
    )

    # Compute W + (alpha / rank) * A @ B.
    scale = alpha / rank
    adapted_kernel = value + scale * (lora_a @ lora_b)

    return adapted_kernel, params

  # Replace get_param with our LoRA version.
  layer.get_param = lora_get_param

  # Store config for merge/removal.
  layer._lora_original_get_param = original_get_param


def remove_lora(layer: bx.Linear):
  """Removes LoRA from a layer, restoring original behavior."""
  if hasattr(layer, '_lora_original_get_param'):
    layer.get_param = layer._lora_original_get_param
    del layer._lora_original_get_param


print('LoRA functions defined!')

LoRA functions defined!


## 4. Apply LoRA to Selected Layers

Let's apply LoRA to the hidden layers but not the output layer.

In [6]:
# Apply LoRA to all Linear layers except the output layer.
for path, module in graph.walk():
  if isinstance(module, bx.Linear) and path[-1] != 'output':
    apply_lora(module, rank=4)
    print(f'Applied LoRA to {path}')

Applied LoRA to ('net', 'mlp', 'hidden0')
Applied LoRA to ('net', 'mlp', 'hidden1')


## 5. Initialize LoRA Parameters

Now we need to run a forward pass to create the LoRA parameters. Since params are locked, we first unlock them.

In [7]:
# Unlock params to allow new parameters.
params = params.unlocked()

# Run forward pass to initialize LoRA params.
_, params = model(params, x)

# Lock again.
params = params.locked()

print(f'Total parameters after LoRA: {len(params)}')
print('\nNew LoRA parameters:')
for path, param in params.items():
  if 'lora' in path[-1]:
    print(f'  {path}: {param.value.shape}')

Total parameters after LoRA: 12

New LoRA parameters:
  ('net', 'mlp', 'hidden0', 'lora_a'): (16, 4)
  ('net', 'mlp', 'hidden0', 'lora_b'): (4, 64)
  ('net', 'mlp', 'hidden1', 'lora_a'): (64, 4)
  ('net', 'mlp', 'hidden1', 'lora_b'): (4, 32)


## 6. Freeze Base Weights

For LoRA training, we want to freeze the original weights and only train the LoRA parameters.

In [8]:
def freeze_base_weights(params, layers):
  """Freezes all non-LoRA parameters in the given layers."""
  for layer in layers:
    # Freeze kernel and bias.
    params = layer.set_param(params, 'kernel', None, trainable=False)
    if layer.use_bias:
      params = layer.set_param(params, 'bias', None, trainable=False)
  return params


# Freeze all base weights.
params = freeze_base_weights(params, model.layers)

print('Base weights frozen!')
print('\nTrainable parameters:')
for path, param in params.items():
  if param.trainable:
    print(f'  {path}: {param.value.shape}')

Base weights frozen!

Trainable parameters:
  ('net', 'mlp', 'hidden0', 'lora_a'): (16, 4)
  ('net', 'mlp', 'hidden0', 'lora_b'): (4, 64)
  ('net', 'mlp', 'hidden1', 'lora_a'): (64, 4)
  ('net', 'mlp', 'hidden1', 'lora_b'): (4, 32)


## 7. Training with LoRA

Now we can train! Only LoRA parameters will receive gradients.

In [9]:
# Generate some dummy training data.
x_train = jax.random.normal(jax.random.key(0), (32, 16))
y_train = jax.random.normal(jax.random.key(1), (32, 10))

# Save params before training for comparison later.
params_before_training = jax.tree.map(lambda x: x.copy(), params)


@jax.jit(donate_argnames='params')
def train_step(params, x, y):
  # Split into trainable (LoRA) and non-trainable (base + RNG).
  trainable, non_trainable = params.split()

  def loss_fn(t, nt):
    full_params = t.merge(nt)
    pred, new_params = model(full_params, x)
    _, new_nt = new_params.split()
    return jnp.mean((pred - y) ** 2), new_nt

  grads, new_non_trainable = jax.grad(loss_fn, has_aux=True)(
      trainable, non_trainable
  )

  # SGD update.
  new_trainable = jax.tree.map(lambda w, g: w - 0.01 * g, trainable, grads)

  return new_trainable.merge(new_non_trainable)


# Compute initial loss.
pred, _ = model(params, x_train)
initial_loss = jnp.mean((pred - y_train) ** 2)
print(f'Initial loss: {initial_loss:.4f}')

# Train for a few steps.
for step in range(100):
  params = train_step(params, x_train, y_train)

# Compute final loss.
pred, _ = model(params, x_train)
final_loss = jnp.mean((pred - y_train) ** 2)
print(f'Final loss: {final_loss:.4f}')
print(f'Loss reduced by {(1 - final_loss/initial_loss) * 100:.1f}%')

Initial loss: 1.2383
Final loss: 0.9382
Loss reduced by 24.2%


## 8. Verify Only LoRA Weights Changed

Let's verify that only the LoRA parameters were updated during training.

In [10]:
# Compare base weights before and after training.
print('Base weights unchanged:')
for path, param in params_before_training.items():
  if 'lora' not in path[-1]:
    match = jnp.allclose(param.value, params[path].value)
    print(f'  {path}: {match}')

Base weights unchanged:
  ('net', 'mlp', 'hidden0', 'bias'): True
  ('net', 'mlp', 'hidden0', 'kernel'): True
  ('net', 'mlp', 'hidden1', 'bias'): True
  ('net', 'mlp', 'hidden1', 'kernel'): True
  ('net', 'mlp', 'output', 'bias'): True
  ('net', 'mlp', 'output', 'kernel'): True
  ('net', 'rng', 'counter'): True
  ('net', 'rng', 'seed'): True


## 9. Merging LoRA Weights (Optional)

For inference, you can merge LoRA weights into the base weights to avoid the extra computation.

In [ ]:
def merge_lora_weights(params, layer):
  """Merges LoRA weights into base weights for efficient inference."""
  if not hasattr(layer, '_lora_original_get_param'):
    return params

  # get_param returns the adapted kernel (W + A @ B) while LoRA is active.
  merged_kernel, _ = layer.get_param(params, 'kernel')

  # Update the base kernel with the merged value.
  return layer.set_param(params, 'kernel', merged_kernel)


# Store output before merging.
out_with_lora, _ = model(params, x_train[:1])

# Merge LoRA weights into base kernels.
merged_params = params
for layer in model.layers:
  merged_params = merge_lora_weights(merged_params, layer)

# Remove LoRA from layers.
for layer in model.layers:
  remove_lora(layer)


# Remove LoRA params from the container.
def is_lora_param(path, param):
  return 'lora' in path[-1]


lora_params, merged_params = merged_params.split(is_lora_param)

# Test that output is the same.
out_merged, _ = model(merged_params, x_train[:1])
outputs_match = jnp.allclose(out_with_lora, out_merged, atol=1e-5)
print(f'Outputs match after merge: {outputs_match}')
print(f'Parameters after merge: {len(merged_params)} (was {len(params)})')

Outputs match after merge: True
Parameters after merge: 8 (was 12)


## Summary

This notebook demonstrated how to implement LoRA fine-tuning with blox:

1. **Graph traversal** - Used `Graph.walk()` to find all Linear layers in the model
2. **Monkey-patching** - Wrapped `get_param` to inject the low-rank adaptation `W + A @ B`
3. **Parameter freezing** - Marked base weights as non-trainable with `set_param(..., trainable=False)`
4. **Efficient training** - Used `params.split()` to separate trainable LoRA params from frozen base weights
5. **Weight merging** - Combined LoRA weights back into base weights for efficient inference

### References

- [LoRA: Low-Rank Adaptation of Large Language Models](https://arxiv.org/abs/2106.09685)
- [Practical Tips for Finetuning LLMs Using LoRA](https://magazine.sebastianraschka.com/p/practical-tips-for-finetuning-llms)